# Exploratory analysis - BTC liquidation imbalance

Reads only what `main.py` saved to disk (nothing here is the source of truth). Re-run `python main.py` first.

In [1]:
import pandas as pd, numpy as np, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
T = ROOT / 'results' / 'tables'
pd.set_option('display.width', 200, 'display.max_columns', 40)

## Core summary (Table 17), ±10% window, map M1

In [2]:
core = pd.read_csv(T / 'T17_core_summary_M1_w10.csv')
core[['bucket','N_events','large_pool_up_N','large_pool_down_N','P_large_first_1pct','P_large_first_2pct','P_large_first_3pct','P_large_first_5pct','median_DAR_24h','mean_DAR_24h','mean_DAR_24h_cluster_ci','p_value_DAR24_cluster','sample_flag']]

,bucket,N_events,large_pool_up_N,large_pool_down_N,P_large_first_1pct,P_large_first_2pct,P_large_first_3pct,P_large_first_5pct,median_DAR_24h,mean_DAR_24h,mean_DAR_24h_cluster_ci,p_value_DAR24_cluster,sample_flag
0,1-1.25,1646,831,815,0.498783,0.512210,0.504750,0.505410,0.000372,0.000775,"[-0.0008, 0.0022]",0.3150,reasonable
1,1.25-1.5,1772,887,885,0.517494,0.519569,0.504706,0.504686,-0.000782,0.000163,"[-0.0010, 0.0015]",0.8035,reasonable
2,1.5-2,1567,773,794,0.514359,0.507033,0.507237,0.494862,-0.001674,-0.001416,"[-0.0028, -0.0001]",0.0380,reasonable
3,2-2.5,874,406,468,0.519451,0.490847,0.497055,0.499322,-0.002313,-0.000403,"[-0.0028, 0.0021]",0.7360,reasonable
4,2.5-3,501,219,282,0.496000,0.502000,0.498965,0.486301,-0.004720,-0.003671,"[-0.0076, -0.0003]",0.0495,reasonable
5,3-3.5,295,109,186,0.467797,0.464407,0.466667,0.486381,-0.004134,-0.003531,"[-0.0072, 0.0004]",0.0635,reasonable
6,3.5-4,193,68,125,0.502591,0.507772,0.486772,0.502924,-0.004151,-0.001321,"[-0.0063, 0.0038]",0.6180,reasonable
7,4-5,145,46,99,0.455172,0.455172,0.454545,0.496124,-0.008414,-0.004572,"[-0.0106, 0.0023]",0.1455,reasonable
8,5-7.5,93,37,56,0.451613,0.387097,0.402174,0.435294,-0.004698,-0.006100,"[-0.0142, 0.0022]",0.1385,small sample
9,>7.5,49,16,33,0.428571,0.326531,0.244898,0.318182,-0.023260,-0.011890,"[-0.0270, 0.0039]",0.1505,small sample


## Asymmetric table (Table 18)

In [3]:
a = pd.read_csv(T / 'T18_asymmetric_M1_w10.csv')
a[['large_dir','bucket','N_events','P_large_first_2pct','P2_benchmark','median_DAR_24h','mean_DAR_24h','p_value_DAR24_cluster']]

,large_dir,bucket,N_events,P_large_first_2pct,P2_benchmark,median_DAR_24h,mean_DAR_24h,p_value_DAR24_cluster
0,DOWN,1-1.25,815,0.506158,0.494337,0.000379,-0.000485,0.6780
1,DOWN,1.25-1.5,885,0.515873,0.494337,-0.002133,-0.000976,0.3285
2,DOWN,1.5-2,794,0.479140,0.494337,-0.002600,-0.002484,0.0305
3,DOWN,2-2.5,468,0.506410,0.494337,-0.001900,-0.001201,0.4135
4,DOWN,2.5-3,282,0.510638,0.494337,-0.006153,-0.005052,0.0205
5,DOWN,3-3.5,186,0.473118,0.494337,-0.005282,-0.006161,0.0145
6,DOWN,3.5-4,125,0.464000,0.494337,-0.007852,-0.008550,0.0010
7,DOWN,4-5,99,0.444444,0.494337,-0.009913,-0.010927,0.0010
8,DOWN,5-7.5,56,0.446429,0.494337,-0.003850,-0.011569,0.0140
9,DOWN,>7.5,33,0.363636,0.494337,-0.021029,-0.015297,0.0455


## Parameter stability: share of significant cells in the full matrix

In [4]:
F = pd.read_csv(T / 'results_matrix_first_passage_M1.csv')
D = pd.read_csv(T / 'results_matrix_DAR_M1.csv')
print('first-passage cells p<0.05:', (F.p_cluster_vs_50 < 0.05).mean().round(3), ' q<0.10:', (F.q_fdr < 0.10).mean().round(3))
print('DAR cells p<0.05:', (D.p_cluster < 0.05).mean().round(3), ' q<0.10:', (D.q_fdr < 0.10).mean().round(3))
F.stability.value_counts()

first-passage cells p<0.05: 0.12  q<0.10: 0.024
DAR cells p<0.05: 0.1  q<0.10: 0.018


stability
stable (neighbour supports)     60
likely unstable / data-mined    24
Name: count, dtype: int64

## Panel regressions - continuous signed imbalance, all specs

In [5]:
R = pd.read_csv(T / 'panel_regressions_all_specs.csv')
R[R.model.str.contains(r'sli \+ size') | R.model.str.contains(r'\|sli\| \+ size')][['model','spec','N','coef_sli','z_sli','coef_abs_sli','z_abs_sli','incremental_R2_of_imbalance']]

,model,spec,N,coef_sli,z_sli,coef_abs_sli,z_abs_sli,incremental_R2_of_imbalance
1,DIR fret_240 ~ sli + size + controls,w3,12700.0,-0.000137,-1.204600,0.000072,0.429960,0.000176
4,DIR fret_1440 ~ sli + size + controls,w3,12695.0,0.000137,0.393820,-0.000072,-0.156700,0.000030
7,DIR fret_4320 ~ sli + size + controls,w3,12683.0,0.000245,0.337423,-0.000043,-0.048114,0.000033
10,DIR logit up_first_2pct ~ sli + size + controls,w3,12645.0,-0.024333,-0.968171,-0.026829,-0.742933,0.000154
11,VOL log fwd_rv_24h ~ |sli| + size + controls(i...,w3,12696.0,NaN,NaN,0.043889,7.794860,0.004091
...,...,...,...,...,...,...,...,...
169,DIR fret_4320 ~ sli + size + controls,exp50,12683.0,-0.000581,-0.417901,0.000364,0.237605,0.000064
172,DIR logit up_first_2pct ~ sli + size + controls,exp50,12645.0,-0.009222,-0.216346,-0.065019,-1.147270,0.000008
173,VOL log fwd_rv_24h ~ |sli| + size + controls(i...,exp50,12696.0,NaN,NaN,0.085765,9.710070,0.006797
175,MAG log range_24h ~ |sli| + size + controls(in...,exp50,12695.0,NaN,NaN,0.058271,4.434290,0.002132


## Walk-forward

In [6]:
pd.read_csv(T / 'walk_forward.csv')

,target,model,N_train,N_test,coef_sli_train,AUC_test,logloss_test,coef_sli_test,p_sli_test_HAC,OOS_R2,sign_hit_rate_test,coef_abs_sli_train,spec
0,up_first_2pct,constant,8885,3809,NaN,0.500000,0.693278,NaN,NaN,NaN,NaN,NaN,w5
1,up_first_2pct,controls_only,8842,3803,NaN,0.517739,0.694055,NaN,NaN,NaN,NaN,NaN,w5
2,up_first_2pct,sli_only,8885,3809,0.061972,0.480048,0.694368,NaN,NaN,NaN,NaN,NaN,w5
3,up_first_2pct,sli+controls,8842,3803,0.063710,0.507761,0.696356,NaN,NaN,NaN,NaN,NaN,w5
4,fret_1440,sli_only,8907,3837,-0.000901,NaN,NaN,-0.000525,0.507982,-0.000224,0.512640,NaN,w5
5,fret_1440,sli+controls,8864,3831,-0.000696,NaN,NaN,0.000290,0.834372,-0.003198,0.491256,NaN,w5
6,fret_1440,controls_only,8864,3831,NaN,NaN,NaN,NaN,NaN,-0.002283,0.491778,NaN,w5
7,fret_4320,sli_only,8907,3825,-0.000792,NaN,NaN,-0.001731,0.353598,0.000736,0.510588,NaN,w5
8,fret_4320,sli+controls,8864,3819,-0.001780,NaN,NaN,-0.000924,0.708321,-0.009759,0.507463,NaN,w5
9,fret_4320,controls_only,8864,3819,NaN,NaN,NaN,NaN,NaN,-0.002086,0.499345,NaN,w5
